In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, QuantileDiscretizer
from pyspark.ml import Pipeline
import pickle
from pathlib import Path
import json

In [0]:
# ============================================================
# Optimized spark dataframe value transformation helper
# ============================================================

def create_preprocessing_pipeline(df, cat_cols, num_cols, id_col):
    """
    Create a single MLlib pipeline for all transformations.
    This is much more efficient than sequential transformations.
    """
    stages = []
    
    # Categorical encoding using StringIndexer (MLlib)
    str_cols = cat_cols + [id_col]
    indexers = StringIndexer(inputCols=str_cols, outputCols=[c+'_idx' for c in str_cols], handleInvalid="keep")

    # Numerical normalization using QuantileDiscretizer with many bins
    # This approximates quantile normalization but is much faster
    discretizers = QuantileDiscretizer(numBuckets=1000,
                                    inputCols=num_cols,
                                    outputCols=[c + "_norm" for c in num_cols],
                                    handleInvalid="keep",
                                    relativeError=0.001  # Balance between accuracy and speed
                                    )

    # Create and fit pipeline
    pipeline = Pipeline(stages=[indexers, discretizers])

    return pipeline


def process_spark_dataframes_optimized(df, id_col, cat_cols, num_cols, partition_col, fitted_pipeline=None):
    """
    Optimized processing using MLlib pipeline.
    
    Args:
        fitted_pipeline: If provided, use this fitted pipeline (for inference)
                        If None, fit a new pipeline (for training)
    """
    
    # Apply or fit pipeline
    if fitted_pipeline is None:
        pipeline = create_preprocessing_pipeline(df, cat_cols, num_cols, id_col)
        fitted_pipeline = pipeline.fit(df)
    
    df_transformed = fitted_pipeline.transform(df)
    
    # Shift indices by 1 to reserve 0 for padding (do this in bulk)
    idx_cols = [id_col + "_idx"] + [c + "_idx" for c in cat_cols]
    for idx_col in idx_cols:
        df_transformed = df_transformed.withColumn(
            idx_col, 
            (F.col(idx_col) + 1).cast("int")
        )
    
    # Normalize the discretized values to [0, 1] range
    for col in num_cols:
        df_transformed = df_transformed.withColumn(
            col + "_norm",
            F.col(col + "_norm") / 1000.0
        )
    
    # Select only needed columns
    select_cols = [
        id_col,
        id_col + '_idx',
        *cat_cols,
        *[c + "_idx" for c in cat_cols],
        *num_cols,
        *[c + "_norm" for c in num_cols],
        partition_col
    ]
    
    df_proc = df_transformed.select(*select_cols)
    
    return df_proc, fitted_pipeline


def save_metadata_optimized(user_df, product_df, user_id_col, product_id_col, 
                           user_cat_cols, product_cat_cols, user_num_cols, 
                           product_num_cols, metadata_dir, user_pipeline, product_pipeline):
    """
    Optimized metadata saving with bulk operations.
    """
    # Save pipelines directly - they contain all transformation logic
    user_pipeline.write().overwrite().save(f"{metadata_dir}/user_pipeline")
    product_pipeline.write().overwrite().save(f"{metadata_dir}/product_pipeline")
    
    # Collect unique IDs using aggregation
    known_user_ids = set([row[user_id_col] for row in 
                         user_df.select(user_id_col).distinct().toLocalIterator()])
    known_product_ids = set([row[product_id_col] for row in 
                         product_df.select(product_id_col).distinct().toLocalIterator()])
    
    # Collect categorical values
    known_user_cat_values = {}
    for col in user_cat_cols:
        known_user_cat_values[col] = set([row[col] for row in 
                                         user_df.select(col).distinct().toLocalIterator()])
    
    known_product_cat_values = {}
    for col in product_cat_cols:
        known_product_cat_values[col] = set([row[col] for row in 
                                         product_df.select(col).distinct().toLocalIterator()])
    
    # Save metadata
    Path(metadata_dir).mkdir(parents=True, exist_ok=True)
    pickle.dump(known_user_ids, open(Path(metadata_dir) / 'known_user_ids.pkl', 'wb'))
    pickle.dump(known_product_ids, open(Path(metadata_dir) / 'known_product_ids.pkl', 'wb'))
    pickle.dump(known_user_cat_values, open(Path(metadata_dir) / 'known_user_cat_values.pkl', 'wb'))
    pickle.dump(known_product_cat_values, open(Path(metadata_dir) / 'known_product_cat_values.pkl', 'wb'))


def load_spark_dataframes(spark, db, table_names, run_date, sample_window=180):
    """Load user and product dataframes"""
    
    user_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['mmc_user_features']}
        WHERE partition_date BETWEEN date_sub('{run_date}', {sample_window}) AND '{run_date}'
    """)
    
    product_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['mmc_product_features']}
        WHERE partition_date BETWEEN date_sub('{run_date}', {sample_window}) AND '{run_date}'
    """)
     
    # Cache these dataframes as they'll be used multiple times
    user_df.cache()
    product_df.cache()
    
    return user_df, product_df


def save_spark_dataframes(spark, user_df, product_df, partition_col, db, table_names, parquet_dir):
    # save processed feature data as delta tables
    user_df.write.format("delta") \
        .partitionBy(partition_col) \
        .mode("overwrite") \
        .saveAsTable("{db}.{mmc_user_processed_features}".format(db=db, mmc_user_processed_features=table_names['mmc_user_processed_features']))
    product_df.write.format("delta") \
        .partitionBy(partition_col) \
        .mode("overwrite") \
        .saveAsTable("{db}.{mmc_product_processed_features}".format(db=db, mmc_product_processed_features=table_names['mmc_product_processed_features']))
    
    return None


def process_user_product_spark_dataframes_optimized(
    user_df, product_df,
    user_id_col, product_id_col,
    user_cat_cols, user_num_cols,
    product_cat_cols, product_num_cols,
    partition_col,
    spark,
    db,
    table_names,
    parquet_dir,
    metadata_dir
):
    """
    Optimized processing with MLlib pipelines and efficient joins.
    """
    
    # Process User Table with pipeline
    user_df_proc, user_pipeline = process_spark_dataframes_optimized(
        user_df, user_id_col, user_cat_cols, user_num_cols, partition_col
    )
    
    # Process product Table with pipeline
    product_df_proc, product_pipeline = process_spark_dataframes_optimized(
        product_df, product_id_col, product_cat_cols, product_num_cols, partition_col
    )
    
    # Repartition
    user_df_proc = user_df_proc.repartition(user_id_col, partition_col)
    product_df_proc = product_df_proc.repartition(product_id_col, partition_col)
    
    # Cache processed dataframes before joins
    user_df_proc.cache()
    product_df_proc.cache()
    
    # Save processed feature data
    save_spark_dataframes(spark, user_df_proc, product_df_proc, partition_col, db, table_names, parquet_dir)

    # Save metadata
    save_metadata_optimized(
        user_df_proc, product_df_proc, 
        user_id_col, product_id_col, 
        user_cat_cols, product_cat_cols, 
        user_num_cols, product_num_cols, 
        metadata_dir,
        user_pipeline, product_pipeline
    )
    
    # Collect statistics efficiently using single pass aggregations
    stats_df = user_df_proc.agg(
        F.countDistinct(user_id_col).alias("n_users"),
        *[F.countDistinct(cat_col + '_idx').alias(f"user_cat_{i}") 
          for i, cat_col in enumerate(user_cat_cols)]
    )
    user_stats = stats_df.first()
    
    stats_df = product_df_proc.agg(
        F.countDistinct(product_id_col).alias("n_products"),
        *[F.countDistinct(cat_col + '_idx').alias(f"product_cat_{i}") 
          for i, cat_col in enumerate(product_cat_cols)]
    )
    product_stats = stats_df.first()
    
    # Unpersist cached dataframes
    user_df_proc.unpersist()
    product_df_proc.unpersist()
    user_df.unpersist()
    product_df.unpersist()
    
    return {
        'n_users': user_stats['n_users'],
        'n_products': product_stats['n_products'],
        'user_cat_dims': [user_stats[f'user_cat_{i}'] for i in range(len(user_cat_cols))],
        'product_cat_dims': [product_stats[f'product_cat_{i}'] for i in range(len(product_cat_cols))],
        'user_num_dim': len(user_num_cols),
        'product_num_dim': len(product_num_cols),
        'parquet_dir': parquet_dir,
        'metadata_dir': metadata_dir
    }

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
data_args = dict(
        user_id_col='user_id', 
        product_id_col='product_id',
        user_cat_cols=["platform"],
        user_num_cols=['store_visits_30d',
        'store_visits_90d',
        'pdp_views_30d',
        'pdp_views_90d',
        'tot_items_ordered_30d',
        'tot_items_ordered_90d',
        'tot_items_ordered_180d',
        'app_items_ordered_30d',
        'app_items_ordered_90d',
        'app_items_ordered_180d',
        'coupons_owned_30d',
        'coupons_owned_90d',
        'coupons_owned_180d',
        'coupons_redeemed_30d',
        'coupons_redeemed_90d',
        'coupons_redeemed_180d'],
        product_cat_cols=["product_type"], 
        product_num_cols=['card_uv_imps_30d',
        'card_uv_imps_90d',
        'pdp_uv_imps_30d',
        'pdp_uv_imps_90d',
        'tot_order_cnt_30d',
        'tot_order_cnt_90d',
        'app_order_cnt_30d',
        'app_order_cnt_90d',
        'avg_order_msrp',
        'avg_order_baseprice_60d',
        'avg_trans_price_30d',
        'avg_trans_price_90d'],
        partition_col='partition_date'
)

sample_window = model_config['feature_sample_window']
parquet_dir = model_config['TRAIN_PARQUET_PATH']
metadata_dir = model_config['METADATA_PATH']

In [0]:
# Load Data
user_df, product_df = load_spark_dataframes(
    spark, db, table_names, run_date, sample_window
)

In [0]:
# Process Data
metadata = process_user_product_spark_dataframes_optimized(
    user_df, product_df,
    spark=spark, 
    db=db, 
    table_names=table_names,
    parquet_dir=parquet_dir, 
    metadata_dir=metadata_dir, 
    **data_args
)

print("Processing complete!")
print(f"Number of users: {metadata['n_users']}")
print(f"Number of products: {metadata['n_products']}")

In [0]:
pickle.dump(metadata, open(Path(metadata_dir) / 'metadata_stats.pkl', 'wb'))